# Beyond Default Prediction: A Risk-Adjusted Approach to Loan Optimization

---

## Team Members

Point of Contact: Ashish Rajeev Chaudhary ( AshishRChaudhary )

Hazem Ani (hazemani)  
Kasem Chaisathitwanit (LuckyFriend)  
Prerna Bharti (PrernaBharti28)

## Introduction

Lending institutions have to regularly decide which applicants are likely to repay their loans and which ones might default. These decisions have real financial consequences and also affect who gets access to credit. Most lenders already use models to estimate default risk, but these tools tend to focus on minimizing losses rather than looking at how credit could be extended more responsibly. This matters especially for underserved borrowers (people with limited credit histories), higher debt-to-income ratios, or shorter employment records who often get denied under traditional approval rules. Our project focuses on a specific stakeholder: a mid-size lender that wants to expand credit access for these underserved borrowers in a responsible way, without putting its financial stability at risk.

To address this need, we built an XGBoost classification model using historical loan data to estimate each applicant's probability of default. Instead of treating the prediction as the end goal, we fed the estimated default probabilities into a financial modeling and policy simulation framework. This let us look at how changing loan sizes, interest rates, and terms affects expected returns and potential losses for different types of borrowers. We defined underserved borrowers as those with a debt-to-income ratio above 0.3 or an employment length under two years, and we specifically looked at how different lending policies could help extend credit to this group while keeping risk at an acceptable level.

The connection between our predictive model and the stakeholder's need is fairly direct. The default probabilities from the XGBoost model let the lender go beyond simple approve-or-deny decisions and instead evaluate the expected financial outcome of lending different amounts to different borrower profiles. Through our policy simulation, we were able to show how a lender could find loan structures that improve profitability while also opening up access for borrowers who would probably be turned down under conventional criteria. Overall, the project tries to bridge the gap between risk prediction and practical lending strategy, giving the stakeholder a data-driven way to make credit decisions that are both financially sound and more inclusive.

## Literature Review

Loan default prediction is generally treated as a binary classification problem. Traditional approaches tend to use logistic regression because it is easy to interpret and widely accepted in financial settings. More recent work has shown that machine learning models like random forests and gradient boosting can improve prediction performance, particularly when measured by ROC-AUC.<sup><a href="#ref1">[1]</a></sup> However, most of these studies focus mainly on improving accuracy rather than looking at the financial impact of lending decisions.

For a mid-size lender that wants to expand access to underserved borrowers, this gap creates a real problem. The lender has to balance two competing goals: giving more people access to credit while also protecting the institution from financial losses. In practice, lenders use an actuarial framework that combines Probability of Default (PD), Loss Given Default (LGD), and Exposure at Default (EAD) to calculate expected loss.<sup><a href="#ref2">[2]</a></sup> But even with this framework, lending decisions are often reduced to simple cutoff rules where borrowers are either approved or denied based on their predicted risk score. Loan size is usually treated as fixed, and there is not much discussion in the literature about how adjusting loan amounts could help balance risk and return more effectively.

We chose XGBoost as our classification model for several reasons. Prior research has consistently shown that gradient boosting methods perform well on structured tabular data like loan records, and XGBoost in particular handles mixed feature types and class imbalance effectively, both of which are common in credit risk datasets. It also provides feature importance scores, which help us understand what factors are driving default risk — something that matters to a lender who needs to explain and trust the model's outputs. We also tested logistic regression as a baseline, and XGBoost achieved a higher ROC-AUC score, which confirmed that it was the stronger choice for our dataset.

While prior research provides strong methods for predicting default, fewer studies connect those predictions to loan-size optimization under clear risk tolerance assumptions. Our project aimed to bridge that gap by combining XGBoost-based default prediction with expected financial value calculations and policy simulation, allowing us to move beyond simple approve-or-deny decisions and explore more flexible lending strategies for underserved borrowers.

## Data and Methods

### Data

**Dataset Source**  
The dataset used in this project comes from Lending Club, a peer-to-peer lending platform that provides public loan performance data for research and analysis. The data was accessed through Kaggle, Lending Club Loan Data.  
Data Link: https://www.kaggle.com/datasets/adarshsng/lending-club-loan-data-csv?select=loan.csv.  

Lending Club datasets have been widely used in academic and industry research related to credit risk modeling, which gives us confidence that the data is credible and suitable for this type of analysis. Because the dataset reflects real financial transactions rather than simulated data, it provides a realistic environment for building and testing predictive models of default risk.

**Dataset Overview**  
The original dataset contains approximately 2.26 million loan records and over 140 variables covering borrower credit history, loan characteristics, payment behavior, and credit account activity. After cleaning, romoving columns with a large amount of missing values, removing redundant columns, dropping high-cardinality features like employer title and borrower state, and filtering to only completed loans, we ended up with 1,303,886 rows and 59 features. These features include a mix of numerical variables (such as loan amount, annual income, debt-to-income ratio, and interest rate) and categorical variables (such as loan grade, home ownership status, and loan purpose). It is worth noting that some of the 59 features contain post-origination data that was retained for the financial modeling portion of the project. Before training the prediction model, these columns were dropped to prevent future data leakage.

**Target Variable and Class Balance**  
We defined the target variable by mapping loan_status values to a binary outcome. Loans marked as "Fully Paid" or "Does not meet the credit policy. Status: Fully Paid" were labeled as non-default (0), while loans marked as "Charged Off," "Default," or "Does not meet the credit policy. Status: Charged Off" were labeled as default (1). Loans still in progress or in other statuses were excluded. After this filtering, the dataset had a default rate of approximately 20.08%, with 1,042,005 non-default loans and 261,881 defaults. This means the classes are imbalanced but not severely so, which is fairly typical for credit risk datasets.

**Exploratory Data Analysis**  
Before modeling, we explored the dataset and reviewed the metadata documentation to understand the meaning of each variable. This helped us identify redundant or irrelevant features and figure out which variables could contribute to predicting loan default.

We plotted histograms for numerical variables to examine their distributions and detect outliers. One notable case was annual income, which contained extremely large values. These outliers were removed to prevent them from disproportionately influencing the model.

We also computed a correlation matrix and visualized it as a heatmap, which helped us find highly correlated feature pairs. For example, loan_amnt and funded_amnt were very highly correlated, so we removed funded_amnt to reduce redundancy.

![correlation_matrix](correlation_matrix.png)

To better understand how loan characteristics relate to default risk, we also plotted bar graphs showing default rates across different loan size ranges and loan terms, as well as a heatmap showing how default rates vary across combinations of loan size, interest rate, and term. These visualizations helped confirm that factors like higher interest rates and longer terms tend to be associated with higher default rates, which informed both our feature selection and our later financial modeling decisions.

![default_rate_by_terms_and_loan_sizes](default_rate_by_term.png)

![default_rate_heatmap_by_term_rate_loan_sizes](default_rate_heatmap_by_term.png)

### Methods

This section describes the full modeling pipeline, from preprocessing through default prediction to the financial policy simulation. The project consisted of three main stages: data preprocessing, classification modeling (including probability calibration), and financial policy simulation.

>**Preprocessing and Feature Engineering**    

The raw dataset contained over 140 columns, many of which had substantial missing data. We began by dropping all columns with more than 35% missing values. For columns with 2,000 or fewer missing rows, we dropped those rows entirely since they represented a negligible fraction of the 2.26 million records. We also removed the emp_title column due to its extremely high cardinality (over 500,000 unique values), which made it impractical to encode.  

Several features required manual transformation. The term column was converted from a string like "36 months" to an integer. Employment length was mapped from categorical labels (e.g., "< 1 year," "10+ years") to numeric values 0 through 10, with missing values imputed using the median. We created a binary Employed feature based on whether an employment title was present. The earliest_cr_line column, which stored the date of a borrower's first credit line, was converted into a credit_age_years feature representing the number of years since that date.  

To handle outliers, we capped annual income at the 99.9th percentile rather than removing rows, which preserved data while limiting the influence of extreme values. We also removed funded_amnt after the correlation analysis showed it was nearly identical to loan_amnt.  

Before training the prediction model, we dropped 14 columns that contained post-origination information to prevent future data leakage. These included variables like total_rec_prncp, recoveries, collection_recovery_fee, and installment, which are only known after a loan has been issued or resolved. These columns were retained in a separate copy of the dataset for use in the financial modeling stage, where actual loan outcomes were needed to estimate quantities like Loss Given Default.  

For the target variable, we filtered the dataset to include only completed loans. Loans marked as "Fully Paid" or "Does not meet the credit policy. Status: Fully Paid" were labeled as 0 (non-default), while "Charged Off," "Default," and "Does not meet the credit policy. Status: Charged Off" were labeled as 1 (default). After filtering, the dataset contained 1,303,886 rows with a default rate of 20.08%.  

We split the data 70/30 into training and test sets using stratified sampling to preserve class balance. After splitting, we imputed remaining missing values using the mean for numerical columns and the mode for categorical columns. Categorical features were encoded using one-hot encoding, and numerical features were scaled using StandardScaler.

>**Classification Modeling**  

We trained two classification models to predict the probability of loan default: logistic regression and XGBoost.  

For `logistic regression`, we used Scikit-learn's SGDClassifier with log loss, which applies stochastic gradient descent to fit a logistic regression model. We chose SGD over a standard solver because of the large dataset size. To address the class imbalance (approximately 80/20 split), we set class weights to {0: 1, 1: 4}, giving more importance to the minority default class. We evaluated the model using 5-fold stratified cross-validation, measuring both F1 score and ROC-AUC. The logistic regression model achieved a mean cross-validated F1 of 0.422 and a ROC-AUC of 0.713.  

For `XGBoost`, we used XGBClassifier with 300 estimators, a max depth of 6, a learning rate of 0.1, and subsample and colsample_bytree both set to 0.8. To handle class imbalance, we used the scale_pos_weight parameter, setting it to the ratio of non-default to default samples in the training set. We also evaluated this model using 5-fold stratified cross-validation. XGBoost achieved a mean cross-validated F1 of 0.447 and a ROC-AUC of 0.734, outperforming logistic regression on both metrics.  

We also attempted hyperparameter tuning using RandomizedSearchCV and tested SMOTE for oversampling the minority class, but neither approach improved performance over the base XGBoost configuration, so we did not include them in the final pipeline.  

Compare model's performance
| Model | Mean CV F1 | Mean CV ROC-AUC |
|-------|-----------|-----------------|
| Logistic Regression (SGD) | 0.422 | 0.713 |
| XGBoost | 0.447 | 0.734 |

Based on these results, we selected XGBoost as the final model. The top features by importance were interest rate, loan term, employment status, home ownership, and loan purpose — which aligned with our expectations about what drives default risk.  

**Probability Calibration**  
Since we planned to use the predicted default probabilities directly in financial calculations, it was important that these probabilities were well-calibrated, meaning that a predicted probability of, say, 0.20 should correspond to roughly a 20% actual default rate. To achieve this, we applied probability calibration to the XGBoost model using two methods: Platt scaling (sigmoid) and isotonic regression, both fitted with 3-fold cross-validation.  

We compared the calibration curves of the original, sigmoid-calibrated, and isotonic-calibrated models. We also searched for the optimal classification threshold for each calibrated model by evaluating F1 scores across a range of thresholds from 0.1 to 0.9. Both methods achieved very similar performance; the isotonic model achieved a best F1 of 0.452 at a threshold of approximately 0.23, compared to 0.452 for sigmoid at the same threshold. We selected the isotonic-calibrated model for the financial simulation since it had a marginally better F1 score, though the difference was minimal.  

>**Financial Policy Simulation**  

The goal of the financial simulation was to move beyond binary approve-or-deny decisions and explore how adjusting loan sizes could improve profitability while expanding access to underserved borrowers. The simulation had three main components: financial ratio engineering, expected profit calculation, and loan size optimization.  

First, we engineered several financial ratio features to capture borrower affordability and financial strain. These included income-to-loan ratio, loan-to-income ratio, installment-to-income ratio, revolving balance-to-income ratio, and available credit ratio. These ratios helped contextualize each borrower's ability to handle different loan amounts.  

Next, we calculated loan revenue for each borrower using the standard amortization formula, based on their loan amount, interest rate, and term. This gave us the total interest the lender would earn if the borrower fully repaid the loan. We also estimated Loss Given Default (LGD) from the actual data by computing recovery rates for defaulted loans. The recovery rate was calculated as total recovered principal plus recoveries minus collection fees, divided by the loan amount. LGD was then 1 minus the recovery rate. Rather than using a single average LGD, we computed mean LGD by sub-grade, which captured the fact that higher-risk loan grades tend to have higher losses when they default. These grade-level LGD values were mapped back to each borrower based on their sub-grade.  

With the calibrated PD from XGBoost and grade-level LGD, we calculated expected profit for each borrower as:

**Expected Profit = Expected Revenue − Expected Loss**

where:
- Expected Revenue = (1 − PD) × Total Interest
- Expected Loss = PD × LGD × Loan Amount 

For the loan size optimization, we created a simulated grid by applying four loan size multipliers (0.50, 0.75, 1.00, and 1.25) to each borrower's original loan amount. For each simulated loan size, we recalculated the financial ratios and loan revenue. We also adjusted the predicted default probability proportionally to the change in loan size, for example, if a borrower's loan was increased by 25%, their PD was scaled up by the same factor, capped at 1.0. This reflects the assumption that larger loans increase financial strain and therefore default risk, though we acknowledge this is a simplifying linear assumption and the true relationship may be more complex.  

We then introduced a risk-adjusted objective function that combined three components: a risk-adjusted profit measure that discounts expected profit by default probability using a penalty parameter (alpha = 0.5), a Sharpe-style profit-to-risk ratio that rewards high profit relative to default risk, and a composite objective score that further penalizes loans where the installment-to-income ratio is high, reflecting borrower affordability. For each borrower, we selected the loan size multiplier that maximized the objective score, subject to the constraint that expected profit must remain positive.  

Finally, we analyzed outcomes specifically for underserved borrowers, defined as those with a debt-to-income ratio above 30 (with an upper cap of 100 to exclude outliers) or employment length under 2 years. We compared approval rates, average default rates, and expected profits for this group under the traditional approach versus the optimized policy to evaluate whether the simulation could expand credit access while maintaining acceptable risk levels.


### Supporting Files

- `Data_Cleaning_for_Default_Prediction.ipynb`: Prepares the raw Lending Club dataset specifically for default prediction modeling, downloads the data from Kaggle, removes high-null, leakage, or non-predictive columns, encodes and converts features, and handles missing values and outliers to produce a clean dataset for training classification models.

<br>

- `Default_Prediction.ipynb`: Early development notebook that was originally intended to contain both the logistic regression and XGBoost models. We later separated these into dedicated notebooks, so this file is retained to document our initial modeling approach and progress.

<br>

- `logistic_reg.ipynb`: Trains and evaluates the baseline logistic regression model for default prediction, handles preprocessing (imputation, encoding, scaling), applies class weighting to address imbalance, performs 5-fold stratified cross-validation, and evaluates on the test set using F1, ROC-AUC, and confusion matrix.

<br>

- `XGBoost_model.ipynb`: Trains and evaluates the XGBoost classifier for default prediction, compares performance against the logistic regression baseline, analyzes feature importance, and applies probability calibration (Platt scaling and isotonic regression) to produce calibrated default probabilities used in the financial framework.

<br>

- `Data_Prep_for_Policy_Simulation.ipynb`: Prepares the dataset for the financial decision framework, engineers 7 financial ratio features, calculates loan revenue and yield, computes Loss Given Default by sub-grade, and derives expected revenue, loss, and profit for each borrower using calibrated default probabilities from the XGBoost model.

<br>

- `Policy_Simulation.ipynb`: Implements the full loan optimization and policy evaluation pipeline, simulates a grid of loan sizes per borrower, applies a risk-adjusted objective function with affordability constraints to identify optimal loan amounts, compares optimized vs. original lending decisions across risk segments, and evaluates policy impact on underserved borrowers including profit lift, default rate reduction, and approval outcomes.

<br>

- `EDA.ipynb`: Exploratory analysis created late in the project to investigate patterns in default rates across loan sizes, term lengths, and interest rate ranges, used to identify potential policy recommendations for reducing defaults.

## Results

#### Default Prediction Model

The logistic regression baseline achieved a ROC-AUC of 0.715 and a default-class F1-score of 0.41, with a recall of 0.44 and precision of 0.39. Overall accuracy was 75%, though this was largely driven by correct predictions on the majority class.

XGBoost improved on the baseline with a ROC-AUC of 0.733 and a default-class F1-score of 0.45. The most notable gain was in recall, which jumped from 0.44 to 0.68, meaning the model caught a substantially larger share of actual defaults. This came at a tradeoff in precision (0.34 vs 0.39) and overall accuracy (67% vs 75%), but for our use case this was an acceptable exchange. Since the financial framework depends on identifying risky borrowers rather than minimizing false alarms, higher recall is more valuable than higher precision. The confusion matrix for XGBoost showed 208,243 true negatives, 100,359 false positives, 25,507 false negatives, and 53,057 true positives.

| Metric | Logistic Regression | XGBoost |
|--------|---------------------|---------|
| ROC-AUC | 0.715 | 0.733 |
| F1 (default class) | 0.41 | 0.45 |
| Recall (default class) | 0.44 | 0.68 |
| Precision (default class) | 0.39 | 0.34 |
| Accuracy | 0.75 | 0.67 |

![Figure_1](XGBoost_Confusion_Matrix_and_ROC_Curve.png)

#### Feature Importance

Interest rate was by far the strongest predictor of default (importance score 0.196), followed by loan term (0.104). This makes sense in practice — lenders charge higher rates to riskier borrowers, so the rate acts as a proxy for overall borrower risk. Longer terms carrying more default risk is also intuitive, since there is more time for a borrower's financial situation to deteriorate.

Employment status (0.048), home ownership (0.036 for renters, 0.034 for mortgage holders), and small business loan purpose (0.027) were moderate contributors. Financial behavior signals like total credit utilization (0.022), accounts opened in the past 24 months (0.024), and mortgage account count (0.021) rounded out the top ten. These results informed the policy recommendations discussed later.

![Figure_2](XGBoost_Top_20_Features.png)

#### Probability Calibration

The calibration curve shows that the original XGBoost probabilities consistently underestimated actual default rates. Both Platt scaling and isotonic regression pulled the curve closer to the ideal diagonal, with isotonic calibration performing slightly better overall. However, at the default 0.5 classification threshold, both calibrated models saw recall drop sharply (from 0.68 to roughly 0.10–0.11), with F1-scores falling to 0.17–0.18. Since our financial framework uses the continuous probability values rather than hard classifications, this tradeoff was acceptable, and we selected the isotonic-calibrated probabilities for the downstream simulation.

![Figure_3](Calibration_Curve.png)

#### Expected Profit Framework

Before optimization, approximately 85% of loans in the test set had positive expected profit, with a mean expected profit of \$1,077 per borrower and total portfolio profit of approximately \$421 million. LGD ranged from roughly 0.52 for A-grade loans to 0.79 for G-grade loans, with an overall mean of 0.61. Additional visualizations of the expected profit distribution, including breakdowns by sub-grade, loan purpose, and term, can be found in Data_Prep_for_Policy_Simulation.ipynb.

The optimizer selected loan sizes across the full range of multipliers. Of the 391,076 borrowers in the test set, 129,620 were assigned a 0.50x multiplier, 97,867 received 0.75x, 56,801 stayed at 1.00x, and 106,788 were increased to 1.25x. On average, the optimized loan amount was \$11,951 compared to the original \$14,421, a reduction of about \$2,470 per borrower. This overall downward shift reflects the optimizer reducing exposure for higher-risk borrowers, who outnumber low-risk borrowers in the dataset.

The risk-segment breakdown shows the optimizer behaving as expected. Low-risk borrowers (PD ≤ 0.10) saw an average loan increase of 22%, with 89% receiving the 1.25x multiplier. Medium-risk borrowers (PD 0.10–0.20) saw a modest 12% decrease. High-risk borrowers (PD 0.20–0.40) were reduced by 41% on average, with 68% assigned the 0.50x multiplier. Very high-risk borrowers (PD > 0.40) were almost universally reduced to 0.50x (99.8%).


| Risk Segment | Count | Avg Original Loan | Avg Optimized Loan | Avg Change | Avg Expected Profit |
|-----------|-----------|-------------|-----------|------------|-------------|
| Low Risk (<= 0.10) | 101,157 | $13,318 | $16,269 | +22% | $1,459 |
| Medium Risk (0.10-0.20) | 125,818 | $13,173 | $11,858 | −12% | $1,507 |
| High Risk (0.20-0.40) | 128,742 | $15,400 | $9,393 | −41% | $1,860 |
| Very High Risk (> 0.40) | 35,319 | $18,472 | $9,247 | −50% | $2,336 |

![Figure_4](Original_vs_Optimized_Loan_Size.png)

![Figure_5](Loan_Size_Distribution.png)

![Figure_6](Original_vs_Optimized_Loan_Size_by_Risk_Segment.png)

#### Underserved Borrower Analysis

Under a traditional flat approval approach, all 88,584 underserved borrowers would be approved at their original loan amounts, yielding a mean expected profit of $634 per borrower, total subgroup profit of approximately $56.2 million, and an average default rate of 23.7%.

Under the optimized policy, 68,124 of the 88,584 underserved borrowers were approved (76.9%). The remaining 20,460 were filtered out because no simulated loan size produced positive expected profit. Among approved borrowers, mean profit per borrower increased to $1,516, a 139% lift. Total subgroup profit rose to approximately $103.3 million, a gain of $47.1 million. The average default rate among approved borrowers dropped to 18.1%, a 5.6 percentage point reduction. This improvement came from the policy being selective, approving borrowers at right-sized loan amounts while filtering out those who were unprofitable under any configuration.

## Discussion

#### Did We Achieve Our Goals?

On the prediction side, we were partially successful. The XGBoost model meaningfully outperformed the logistic regression baseline, particularly in recall (0.68 vs 0.44), which matters most for our downstream use case since we need to catch risky borrowers rather than minimize false alarms. However, F1-scores for both models remained moderate (0.41–0.45), and none of our improvement attempts (randomized hyperparameter tuning, SMOTE, feature selection for multicollinearity, removing scaling) produced significant gains. This is consistent with what others have found working with the Lending Club dataset, which is widely used as a benchmark for credit risk modeling on Kaggle. Borrower default behavior appears to be inherently difficult to predict from structured application-time data alone, and there seems to be a practical ceiling on how far these models can go with this type of data.

Where the project delivered more clearly was on the financial modeling and policy simulation side. By connecting default probabilities to an expected profit framework and simulating loan size adjustments, we were able to move beyond approve-or-deny decisions and generate borrower-level loan sizing recommendations. The optimizer systematically increased loans for low-risk borrowers and reduced them for high-risk borrowers, which is exactly the kind of behavior the stakeholder would want to see. The underserved borrower analysis showed that the policy could approve 76.9% of that subgroup while improving mean profit by 139% and reducing the default rate by 5.6 percentage points, a result that directly addresses the stakeholder's goal of expanding credit access responsibly.

#### Addressing the Stakeholders Needs

Our stakeholder is a mid-size lender that wants to responsibly expand credit access for underserved borrowers without jeopardizing financial stability. The traditional approach would approve all underserved applicants at their original loan amounts, but this includes a significant number of unprofitable loans that drag down portfolio performance. The optimized policy filters out the 23.1% of borrowers who are unprofitable under any simulated loan size while still approving the remaining 76.9%. For those approved, the policy right-sizes loans to balance profitability and affordability, resulting in total subgroup profit nearly doubling from $56.2 million to $103.3 million.

This is a meaningful result for the stakeholder. Rather than blanket rejections of underserved applicants, the policy identifies which borrowers within that group can be served profitably and at what loan size. It shifts the decision from "should we lend to this person" to "how much should we lend to this person," which is a more nuanced and inclusive approach to credit access.

### Policy Recommendations

The feature importance results and our exploratory analysis provide a basis for translating the optimization into interpretable lending policies. We focus on four key recommendations, each grounded in the model's findings and our data.

#### 1. Dynamic Loan Sizing Based on Borrower Risk & Capacity

This is the core contribution of the project. The model identified DTI, annual income, and credit utilization as meaningful risk drivers, and the optimizer uses these signals to adjust loan amounts per borrower rather than relying on fixed approval thresholds. The results validate this approach: low-risk borrowers received an average 22% increase in loan size (89% assigned the 1.25x multiplier), while high-risk and very high-risk borrowers were systematically scaled down by 41% and 50% respectively. This produced positive expected profit across all segments while concentrating exposure where the risk-return tradeoff is most favorable. The key insight is that loan amount itself appeared as a meaningful predictor of default in the XGBoost model, which supports the simulation's core assumption that default risk increases with loan size and validates the proportional PD adjustment we used.

#### 2. Term-Based Restrictions for Higher-Risk Borrowers

Loan term was the second most important feature in the model (importance score 0.104), and our exploratory analysis reinforces why. The default rate comparison by term shows that 60-month loans consistently default at roughly double the rate of 36-month loans across every loan size bin, for example, in the $10K–$15K range, the 36-month default rate is 15.6% compared to 33.5% for 60-month loans. The heatmap further shows that this gap widens at higher interest rates: a 60-month loan in the 20–25% rate band defaults at 37–44% depending on loan size, compared to 25–39% for 36-month loans in the same band. Our current simulation holds term fixed, but these patterns suggest that restricting high-risk borrowers to 36-month terms could meaningfully reduce default exposure. A lender could implement this as a simple rule, if predicted PD exceeds a threshold, only offer a 36-month term, or integrate term as a second optimization variable alongside loan size.

#### 3. Interest Rate as a Risk Lever

Interest rate was the strongest feature in the model by a wide margin (importance score 0.196), which reflects the fact that lenders already charge higher rates to riskier borrowers. The heatmap makes the risk gradient visible: for 36-month loans in the $10K–$15K range, default rates climb from 7.5% at the 5–10% rate band all the way to 48.6% at the 25–31% band. This tells us that rate is not just a proxy for risk, it is also a lever the lender can adjust. Rather than treating the interest rate as fixed at origination, a lender could pair our loan sizing framework with risk-based pricing: borrowers whose optimized loan size is smaller than their original request could be offered a rate adjustment that compensates for the reduced principal, keeping the loan attractive to the borrower while maintaining the lender's expected return. Our simulation does not currently vary interest rates, but this would be a natural complement to the loan sizing optimization.

#### 4. Borrower Risk Profile Screening

Several moderate-importance features (employment status (0.048), home ownership, credit utilization (0.022), recent inquiries, and past delinquencies) collectively describe the borrower's financial stability. Rather than treating each as a separate policy rule, these can be combined into a borrower risk profile that informs lending decisions. For example, a borrower who is unemployed, has high credit utilization, and recent inquiries presents a very different risk picture than one who only triggers on one of these signals. Our definition of underserved borrowers (DTI > 30 or employment length < 2 years) is one version of this, but a lender could build a more granular scoring system that weights these factors together. The optimizer already implicitly accounts for these signals through the predicted default probability, but making them explicit as interpretable guardrails would help the lender explain and audit lending decisions, which matters for regulatory compliance and stakeholder trust.

These four recommendations are not meant to replace the dynamic optimization framework. They provide interpretable structure around the optimizer's decisions, rules that a lender could implement alongside the simulation, or use independently in settings where running the full optimization is not practical.

## Limitations

While the project achieved its core goals of building a default prediction model and connecting it to a financial policy simulation, there are several limitations worth noting.  

First, class imbalance was a persistent challenge throughout the project. Defaults made up only about 20% of the dataset, which meant the model could achieve seemingly decent accuracy by simply predicting non-default for most borrowers. We addressed this using class weighting and evaluated performance with F1 and ROC-AUC rather than accuracy, but the overall prediction performance remained moderate. The XGBoost model achieved a ROC-AUC of 0.734 and an F1 of 0.447, which is reasonable for credit risk data but leaves room for improvement.  

Second, although we applied probability calibration using both Platt scaling and isotonic regression, these techniques improved probability alignment but yielded only minor gains in overall model performance. We explored several other approaches as well, including randomized hyperparameter tuning, adjusting feature sets to address multicollinearity, SMOTE oversampling, and removing scaling for XGBoost, but none produced significant improvement. Since the financial framework and policy simulation depend heavily on well-calibrated default probabilities, the limited calibration gains mean that the downstream profit and risk estimates should be interpreted with some caution.  

Third, the Lending Club dataset itself presented challenges. The original data contained 145 features, but many columns had more than 30% missing values and had to be dropped. Some of these potentially useful columns such as mths_since_last_delinq (51% missing), mths_since_last_major_derog (74% missing), and mths_since_last_record (84% missing), which could have provided valuable signals about borrower risk history. After cleaning, imputation, and feature engineering, model performance did not improve significantly, which could suggest that borrower default behavior is inherently difficult to predict from structured application-time data alone.  

Fourth, in the policy simulation we adjusted default probabilities proportionally to changes in loan size, if a loan was increased by 25%, the PD was scaled up by the same factor. This is a simplifying linear assumption, and in reality the relationship between loan size and default risk is likely more complex and borrower-specific. Ideally, we would have used the trained XGBoost model to re-predict default probabilities for each simulated loan amount, but given that the model's predictive performance was still moderate, we opted for the proportional adjustment as a practical approximation. Improving the model to support direct re-prediction under different loan scenarios is something we identify as future work.  

Finally, the model was trained and evaluated entirely on historical Lending Club data. Economic conditions, borrower behavior, and lending standards change over time, so both the model's performance and the simulation's conclusions may not generalize well to future lending environments or to other lending institutions with different borrower populations.

## Future Work

There are several directions we would like to explore to improve and extend this project.

First, in our current simulation we adjusted default probabilities proportionally to changes in loan size, which is a simplifying assumption. A more robust approach would be to retrain or use the XGBoost model to directly predict default probabilities for each simulated loan amount, so that the PD estimates reflect the model's learned relationships rather than a linear scaling rule. This would require improving the model's overall predictive performance to a level where we can trust its predictions under modified loan conditions.

Second, our loan size optimization only tested four discrete multipliers (0.50, 0.75, 1.00, and 1.25). A finer or continuous optimization approach could identify more precise loan amounts for each borrower, and expanding beyond the 1.25 upper bound could reveal whether some low-risk borrowers could responsibly handle larger loans than we currently allow.

Third, our simulation held interest rates fixed at their original values while adjusting loan sizes. In practice, lenders adjust interest rates alongside loan amounts based on borrower risk. A future version of the simulation could explore how varying interest rates in combination with loan sizes might help secure the lender against default risk while still offering underserved borrowers more accessible loan terms.

Finally, we would like to build an interactive dashboard that allows users to adjust inputs like loan size, interest rate, and term and immediately see how those changes affect expected profit, default risk, and approval rates for different borrower groups. This would make the simulation results more accessible and practical for a lending stakeholder to use in real decision-making.  


## References
<a id="ref1"></a> [1] [Y. Zhou, Loan Default Prediction Based on Machine Learning Methods, Proceedings of the 3rd International Conference on Big Data Economy and Information Management (BDEIM 2022)](https://eudl.eu/doi/10.4108/eai.2-12-2022.2328740)

<a id="ref2"></a> [2] [Expected Loss (EL): Overview, How to Calculate, Importance in Credit Risk Management](https://corporatefinanceinstitute.com/resources/career-map/sell-side/risk-management/expected-loss-definition-calculation-importance/?utm_source=chatgpt.com)